In [ ]:
# yaml points at the tokenisation pairing outputs - subset those

In [1]:
import pickle, glob, os, scanpy as sc

BASE = "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL"
rowid = pickle.load(open(f"{BASE}/tokenid_to_rowid_all.pkl", "rb"))
names = pickle.load(open(f"{BASE}/token_id_to_genename_all.pkl", "rb"))
special = {"<cls>", "<eos>", "<mask>", "<pad>"}

In [3]:
rk = list(rowid.keys())
nk = list(names.keys())
print("rowid keys :", type(rk[0]), min(rk), max(rk), len(rk))
print("names keys :", type(nk[0]), min(nk), max(nk), len(nk))
print("overlap    :", len(set(rk) & set(nk)))
print("rowid sample:", list(rowid.items())[:5])
print("names sample:", list(names.items())[:5])

rowid keys : <class 'numpy.int64'> 0 18950 1466
names keys : <class 'int'> 0 1465 1466
overlap    : 140
rowid sample: [(np.int64(16308), np.int64(4)), (np.int64(16647), np.int64(5)), (np.int64(780), np.int64(6)), (np.int64(17853), np.int64(7)), (np.int64(7276), np.int64(8))]
names sample: [(4, 'ENSG00000174611'), (5, 'ENSG00000139352'), (6, 'ENSG00000134242'), (7, 'ENSG00000081800'), (8, 'ENSG00000095564')]


In [4]:
print([(i, names[i]) for i in range(6)])

[(0, '<pad>'), (1, '<mask>'), (2, '<cls>'), (3, '<eos>'), (4, 'ENSG00000174611'), (5, 'ENSG00000139352')]


In [7]:
import pickle, glob, os, scanpy as sc

# load the global Geneformer token ID -> local ID - Keys run up to 18950, values are 0–1465
rowid = pickle.load(
    open(
        "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/tokenid_to_rowid_all.pkl",
        "rb",
    )
)

# load ID -> ENSEMBL gene ID (or special token)
names = pickle.load(
    open(
        "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/token_id_to_genename_all.pkl",
        "rb",
    )
)

# building the gene list
# rowid.values() gives the local IDs (0–1465), int(v) casts from np.int64 so they index names (whose keys are plain int) without a KeyError
# sorted - puts them in local ID order = i.e. the ordering that the decoder's output columns follows
gene_order = [names[i] for i in sorted(int(v) for v in rowid.values())][4:]

# sanity check - expected print is 1462 and ENSEMBL IDs
print(len(gene_order), gene_order[:3])

# create the two output directories
os.makedirs(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_1462genes_src",
    exist_ok=True,
)
os.makedirs(
    "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_1462genes_tgt",
    exist_ok=True,
)

for indir, outdir in [
    (
        "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_all_src",
        "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_1462genes_src",
    ),
    (
        "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_all_tgt",
        "/rds/general/project/lms-scott-raw/live/Ada/perturbation_modeling/T_perturb/tokenized_data/adipocytes_subset_obese_WL/h5ad_pairing_1462genes_tgt",
    ),
]:
    for f in sorted(glob.glob(indir + "/*.h5ad")):
        a = sc.read_h5ad(
            f
        )  # it reads the input directiory (the 1974 genes object)
        a = a[
            :, gene_order
        ].copy()  # selection of columns by gene name, in the given order
        a.write_h5ad(
            outdir + "/" + os.path.basename(f)
        )  # and it outputs the subset object
        print(
            os.path.basename(f), a.shape
        )  # print the new shape; expected (1083, 1462) for src.

1462 ['ENSG00000174611', 'ENSG00000139352', 'ENSG00000134242']


/rds/general/user/ap5625/home/miniforge3/envs/perturbgen/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/rds/general/user/ap5625/home/miniforge3/envs/perturbgen/lib/python3.11/site-packages/anndata/_core/anndata.py:1758: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


obese.h5ad (1083, 1462)
1_weightloss.h5ad (1083, 1462)
